# Notebook for calculating hail statistics

- Domain coverage
- Urban coverage
- Maximum hail size

### Import statements

In [6]:
from netCDF4 import Dataset
import wrf
from wrf import getvar, ALL_TIMES, latlon_coords, CoordPair, vertcross, to_np, interpline
import xarray as xr
import numpy as np
import cartopy.crs as crs
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
import matplotlib.cm as cm 
import matplotlib as mpl
import pandas as pd

### Load in datasets

In [7]:
# Function to import files

def dataset_files(setup, run_no, year, file = True):
    year_dict = {
        2017: ['16_00', '15_23', '15_22', '15_21', '15_20', '2017-02'],
        2018: ['18_00', '17_23', '17_22', '17_21', '17_20', '2018-12'],
        2020: ['15_00', '14_23', '14_22', '14_21', '14_20', '2020-12'],
        2024: ['11_00', '10_23', '10_22', '10_21', '10_20', '2024-02'],
    }
    if file:
        t = year_dict[year][run_no-1]
        t_ind = int(t[0:2])
        t2 = str(t_ind+1)+t[2:]
        t3 = str(t_ind+2)+t[2:]
        if run_no == 1:
            dat = [Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}/wrfout_d02_{year_dict[year][5]}-{t}:00:00'),
            Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}/wrfout_d02_{year_dict[year][5]}-{t2}:00:00'),
            Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}/wrfout_d02_{year_dict[year][5]}-{t2}:10:00'),
            Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}/wrfout_d02_{year_dict[year][5]}-{t3}:10:00')]
        else:
            hour = t[3:5]
            dat = [Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}_{str(run_no)}/wrfout_d02_{year_dict[year][5]}-{t}:00:00'),
            Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}_{str(run_no)}/wrfout_d02_{year_dict[year][5]}-{t2}:00:00'),
            Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}_{str(run_no)}/wrfout_d02_{year_dict[year][5]}-{t2}:10:00'),
            Dataset(f'/g/data/li18/em3807_2/WRF_runs/{setup}_{year}_{str(run_no)}/wrfout_d02_{year_dict[year][5]}-{t3}:10:00')]
        return(dat)
    else:
        if run_no == 1:
            dat = Dataset(f'/g/data/li18/em3807_2/WRF_runs/extracted_data/{setup}_{year}_subset.nc')
        else:
            dat = Dataset(f'/g/data/li18/em3807_2/WRF_runs/extracted_data/{setup}_{year}_{run_no}_subset.nc')
        return(dat)

# 2017

data_17_crop = dataset_files('Cropland', 1, 2017)
data_17_crop_2 = dataset_files('Cropland', 2, 2017)
data_17_crop_3 = dataset_files('Cropland', 3, 2017)
data_17_crop_4 = dataset_files('Cropland', 4, 2017)
data_17_crop_5 = dataset_files('Cropland', 5, 2017)

data_17_noeuro = dataset_files('Pre_Euro', 1, 2017, file = False)
data_17_noeuro_2 = dataset_files('Pre_Euro', 2, 2017, file = False)
data_17_noeuro_3 = dataset_files('Pre_Euro', 3, 2017, file = False)
data_17_noeuro_4 = dataset_files('Pre_Euro', 4, 2017, file = False)
data_17_noeuro_5 = dataset_files('Pre_Euro', 5, 2017, file = False)

data_17_nat = dataset_files('Natland', 1, 2017, file = False)
data_17_nat_2 = dataset_files('Natland', 2, 2017, file = False)
data_17_nat_3 = dataset_files('Natland', 3, 2017, file = False)
data_17_nat_4 = dataset_files('Natland', 4, 2017, file = False)
data_17_nat_5 = dataset_files('Natland', 5, 2017, file = False)

data_17_def = dataset_files('NoUCM_WRFDef', 1, 2017, file = False)
data_17_def_2 = dataset_files('NoUCM_WRFDef', 2, 2017, file = False)
data_17_def_3 = dataset_files('NoUCM_WRFDef', 3, 2017, file = False)
data_17_def_4 = dataset_files('NoUCM_WRFDef', 4, 2017, file = False)
data_17_def_5 = dataset_files('NoUCM_WRFDef', 5, 2017, file = False)

data_17_defurb = dataset_files('BEPBEM_WRFDef', 1, 2017, file = False)
data_17_defurb_2 = dataset_files('BEPBEM_WRFDef', 2, 2017, file = False)
data_17_defurb_3 = dataset_files('BEPBEM_WRFDef', 3, 2017, file = False)
data_17_defurb_4 = dataset_files('BEPBEM_WRFDef', 4, 2017, file = False)
data_17_defurb_5 = dataset_files('BEPBEM_WRFDef', 5, 2017, file = False)

data_17_grurb = dataset_files('BEPBEM_GrUrban', 1, 2017, file = False)
data_17_grurb_2 = dataset_files('BEPBEM_GrUrban', 2, 2017, file = False)
data_17_grurb_3 = dataset_files('BEPBEM_GrUrban', 3, 2017, file = False)
data_17_grurb_4 = dataset_files('BEPBEM_GrUrban', 4, 2017, file = False)
data_17_grurb_5 = dataset_files('BEPBEM_GrUrban', 5, 2017, file = False)

data_17_gr = dataset_files('NoUCM_GrUrban', 1, 2017, file = False)
data_17_gr_2 = dataset_files('NoUCM_GrUrban', 2, 2017, file = False)
data_17_gr_3 = dataset_files('NoUCM_GrUrban', 3, 2017, file = False)
data_17_gr_4 = dataset_files('NoUCM_GrUrban', 4, 2017, file = False)
data_17_gr_5 = dataset_files('NoUCM_GrUrban', 5, 2017, file = False)

# 2018

data_18_crop = dataset_files('Cropland', 1, 2018)
data_18_crop_2 = dataset_files('Cropland', 2, 2018)
data_18_crop_3 = dataset_files('Cropland', 3, 2018)
data_18_crop_4 = dataset_files('Cropland', 4, 2018)
data_18_crop_5 = dataset_files('Cropland', 5, 2018, file = False)

data_18_noeuro = dataset_files('Pre_Euro', 1, 2018, file = False)
data_18_noeuro_2 = dataset_files('Pre_Euro', 2, 2018, file = False)
data_18_noeuro_3 = dataset_files('Pre_Euro', 3, 2018, file = False)
data_18_noeuro_4 = dataset_files('Pre_Euro', 4, 2018, file = False)
data_18_noeuro_5 = dataset_files('Pre_Euro', 5, 2018, file = False)

data_18_nat = dataset_files('Natland', 1, 2018, file = False)
data_18_nat_2 = dataset_files('Natland', 2, 2018, file = False)
data_18_nat_3 = dataset_files('Natland', 3, 2018, file = False)
data_18_nat_4 = dataset_files('Natland', 4, 2018, file = False)
data_18_nat_5 = dataset_files('Natland', 5, 2018, file = False)

data_18_def = dataset_files('NoUCM_WRFDef', 1, 2018, file = False)
data_18_def_2 = dataset_files('NoUCM_WRFDef', 2, 2018, file = False)
data_18_def_3 = dataset_files('NoUCM_WRFDef', 3, 2018, file = False)
data_18_def_4 = dataset_files('NoUCM_WRFDef', 4, 2018, file = False)
data_18_def_5 = dataset_files('NoUCM_WRFDef', 5, 2018, file = False)

data_18_defurb = dataset_files('BEPBEM_WRFDef', 1, 2018, file = False)
data_18_defurb_2 = dataset_files('BEPBEM_WRFDef', 2, 2018, file = False)
data_18_defurb_3 = dataset_files('BEPBEM_WRFDef', 3, 2018, file = False)
data_18_defurb_4 = dataset_files('BEPBEM_WRFDef', 4, 2018, file = False)
data_18_defurb_5 = dataset_files('BEPBEM_WRFDef', 5, 2018, file = False)

data_18_grurb = dataset_files('BEPBEM_GrUrban', 1, 2018, file = False)
data_18_grurb_2 = dataset_files('BEPBEM_GrUrban', 2, 2018, file = False)
data_18_grurb_3 = dataset_files('BEPBEM_GrUrban', 3, 2018, file = False)
data_18_grurb_4 = dataset_files('BEPBEM_GrUrban', 4, 2018, file = False)
data_18_grurb_5 = dataset_files('BEPBEM_GrUrban', 5, 2018, file = False)

data_18_gr = dataset_files('NoUCM_GrUrban', 1, 2018, file = False)
data_18_gr_2 = dataset_files('NoUCM_GrUrban', 2, 2018, file = False)
data_18_gr_3 = dataset_files('NoUCM_GrUrban', 3, 2018, file = False)
data_18_gr_4 = dataset_files('NoUCM_GrUrban', 4, 2018, file = False)
data_18_gr_5 = dataset_files('NoUCM_GrUrban', 5, 2018, file = False)

# 2020

data_20_crop = dataset_files('Cropland', 1, 2020)
data_20_crop_2 = dataset_files('Cropland', 2, 2020)
data_20_crop_3 = dataset_files('Cropland', 3, 2020)
data_20_crop_4 = dataset_files('Cropland', 4, 2020)
data_20_crop_5 = dataset_files('Cropland', 5, 2020)

data_20_noeuro = dataset_files('Pre_Euro', 1, 2020, file = False)
data_20_noeuro_2 = dataset_files('Pre_Euro', 2, 2020, file = False)
data_20_noeuro_3 = dataset_files('Pre_Euro', 3, 2020, file = False)
data_20_noeuro_4 = dataset_files('Pre_Euro', 4, 2020, file = False)
data_20_noeuro_5 = dataset_files('Pre_Euro', 5, 2020, file = False)

data_20_nat = dataset_files('Natland', 1, 2020, file = False)
data_20_nat_2 = dataset_files('Natland', 2, 2020, file = False)
data_20_nat_3 = dataset_files('Natland', 3, 2020, file = False)
data_20_nat_4 = dataset_files('Natland', 4, 2020, file = False)
data_20_nat_5 = dataset_files('Natland', 5, 2020, file = False)

data_20_def = dataset_files('NoUCM_WRFDef', 1, 2020, file = False)
data_20_def_2 = dataset_files('NoUCM_WRFDef', 2, 2020, file = False)
data_20_def_3 = dataset_files('NoUCM_WRFDef', 3, 2020, file = False)
data_20_def_4 = dataset_files('NoUCM_WRFDef', 4, 2020, file = False)
data_20_def_5 = dataset_files('NoUCM_WRFDef', 5, 2020, file = False)

data_20_defurb = dataset_files('BEPBEM_WRFDef', 1, 2020, file = False)
data_20_defurb_2 = dataset_files('BEPBEM_WRFDef', 2, 2020, file = False)
data_20_defurb_3 = dataset_files('BEPBEM_WRFDef', 3, 2020, file = False)
data_20_defurb_4 = dataset_files('BEPBEM_WRFDef', 4, 2020, file = False)
data_20_defurb_5 = dataset_files('BEPBEM_WRFDef', 5, 2020, file = False)

data_20_grurb = dataset_files('BEPBEM_GrUrban', 1, 2020, file = False)
data_20_grurb_2 = dataset_files('BEPBEM_GrUrban', 2, 2020, file = False)
data_20_grurb_3 = dataset_files('BEPBEM_GrUrban', 3, 2020, file = False)
data_20_grurb_4 = dataset_files('BEPBEM_GrUrban', 4, 2020, file = False)
data_20_grurb_5 = dataset_files('BEPBEM_GrUrban', 5, 2020, file = False)

data_20_gr = dataset_files('NoUCM_GrUrban', 1, 2020, file = False)
data_20_gr_2 = dataset_files('NoUCM_GrUrban', 2, 2020, file = False)
data_20_gr_3 = dataset_files('NoUCM_GrUrban', 3, 2020, file = False)
data_20_gr_4 = dataset_files('NoUCM_GrUrban', 4, 2020, file = False)
data_20_gr_5 = dataset_files('NoUCM_GrUrban', 5, 2020, file = False)

# 2024

data_24_crop = dataset_files('Cropland', 1, 2024)
data_24_crop_2 = dataset_files('Cropland', 2, 2024)
data_24_crop_3 = dataset_files('Cropland', 3, 2024)
data_24_crop_4 = dataset_files('Cropland', 4, 2024)
data_24_crop_5 = dataset_files('Cropland', 5, 2024)

data_24_noeuro = dataset_files('Pre_Euro', 1, 2024)
data_24_noeuro_2 = dataset_files('Pre_Euro', 2, 2024)
data_24_noeuro_3 = dataset_files('Pre_Euro', 3, 2024)
data_24_noeuro_4 = dataset_files('Pre_Euro', 4, 2024)
data_24_noeuro_5 = dataset_files('Pre_Euro', 5, 2024)

data_24_nat = dataset_files('Natland', 1, 2024)
data_24_nat_2 = dataset_files('Natland', 2, 2024)
data_24_nat_3 = dataset_files('Natland', 3, 2024)
data_24_nat_4 = dataset_files('Natland', 4, 2024)
data_24_nat_5 = dataset_files('Natland', 5, 2024)

data_24_def = dataset_files('NoUCM_WRFDef', 1, 2024)
data_24_def_2 = dataset_files('NoUCM_WRFDef', 2, 2024)
data_24_def_3 = dataset_files('NoUCM_WRFDef', 3, 2024)
data_24_def_4 = dataset_files('NoUCM_WRFDef', 4, 2024)
data_24_def_5 = dataset_files('NoUCM_WRFDef', 5, 2024)

data_24_defurb = dataset_files('BEPBEM_WRFDef', 1, 2024)
data_24_defurb_2 = dataset_files('BEPBEM_WRFDef', 2, 2024)
data_24_defurb_3 = dataset_files('BEPBEM_WRFDef', 3, 2024)
data_24_defurb_4 = dataset_files('BEPBEM_WRFDef', 4, 2024)
data_24_defurb_5 = dataset_files('BEPBEM_WRFDef', 5, 2024)

data_24_grurb = dataset_files('BEPBEM_GrUrban', 1, 2024)
data_24_grurb_2 = dataset_files('BEPBEM_GrUrban', 2, 2024)
data_24_grurb_3 = dataset_files('BEPBEM_GrUrban', 3, 2024)
data_24_grurb_4 = dataset_files('BEPBEM_GrUrban', 4, 2024)
data_24_grurb_5 = dataset_files('BEPBEM_GrUrban', 5, 2024)

data_24_gr = dataset_files('NoUCM_GrUrban', 1, 2024)
data_24_gr_2 = dataset_files('NoUCM_GrUrban', 2, 2024)
data_24_gr_3 = dataset_files('NoUCM_GrUrban', 3, 2024)
data_24_gr_4 = [Dataset('/g/data/li18/em3807_2/WRF_runs/NoUCM_GrUrban_2024_4/wrfout_d02_2024-02-10_21:00:00'),
Dataset('/g/data/li18/em3807_2/WRF_runs/NoUCM_GrUrban_2024_4/wrfout_d02_2024-02-10_22:00:00'),
Dataset('/g/data/li18/em3807_2/WRF_runs/NoUCM_GrUrban_2024_4/wrfout_d02_2024-02-11_21:10:00'),
Dataset('/g/data/li18/em3807_2/WRF_runs/NoUCM_GrUrban_2024_4/wrfout_d02_2024-02-12_21:10:00')]
data_24_gr_5 = dataset_files('NoUCM_GrUrban', 5, 2024)

### Define functions

In [14]:
# Match UTC time to WRF index
def match_tti(dataset, date, time):
    timelist = wrf.extract_times(dataset, timeidx = ALL_TIMES)
    if len(time) ==7:
        timestr = str(date)+"T0"+str(time)+".000000000"
    if len(time) ==8:
        timestr = str(date)+"T"+str(time)+".000000000"
    if isinstance(dataset, list):
        for file in dataset:
            if timestr in str(wrf.extract_times(file, timeidx = ALL_TIMES)):
                file_num = dataset.index(file)
    else:
        file_num = ''
    for idx, t in enumerate(timelist):
        if timestr == str(t):
            ind = idx
            break
    else:
        return print("Time not in dataset")
    return(ind,file_num,f'timeidx: {ind} for {t}')

# Convert UTC to AEST
def utc_to_aest(timestr):
    utc = pd.Timestamp(timestr,tz='UTC')
    aus = utc.tz_convert(tz='Australia/Sydney')
    date = str(aus).split(' ')[0]
    time = str(aus).split(' ')[1][:8]
    wrfstr = date+"T"+time+".000000000"
    aest = time+' '+date
    return(wrfstr,aest,f'AEST is {time} {date}')

# Convert AEST to UTC
def aest_to_utc(date,time):
    datetime = f'{date} {time}'
    austime = pd.Timestamp(datetime,tz='Australia/Sydney')
    utctime = austime.tz_convert(tz='UTC')
    udate = str(utctime).split(' ')[0]
    utime = str(utctime).split(' ')[1][:8] 
    wrfstr = udate+"T"+utime+".000000000"  
    return(wrfstr,[udate, utime])

# Find the number of grid cells with hail
def grid_number(var):
    land = wrf.getvar(data_24_gr, 'LU_INDEX', timeidx=8, method='cat')
    varurban = np.where(land <= 49, 0, var)
    urbanlist = []
    gridnumlist = []
    for i in range (0, len(var)):
        gridnumlist.append(np.count_nonzero(var[i]))
    for i in range (0, len(var)):
        urbanlist.append(np.count_nonzero(varurban[i]))
    gridnum = np.array(gridnumlist)
    urbnum = np.array(urbanlist)
    return(gridnum,urbnum)

# Calculate percentage coverage over different regions
def percentcov(var):
    land = wrf.getvar(data_24_gr, 'LU_INDEX', timeidx=8, method='cat')
    urbland = np.where(land <= 49, 0, land)
    urb = np.where(land <= 49, 0, land)
    variable = np.where(var == 49, 0, var)
    urb[0:140, 0:200] = 0 # area below Sydney
    urb[0:289, 0:150] = 0 # left of Sydney
    urb[205:289, 0:279] = 0 # left of Sydney
    metarea = np.count_nonzero(urb)
    totarea = len(urbland)*len(urbland[0])
    urbarea = np.count_nonzero(urbland)
    vararea = np.count_nonzero(var)
    urbvar = np.where(urbland == 0, 0, var)
    metvar = np.where(urb == 0, 0, var)
    urbvararea = np.count_nonzero(urbvar)
    metvararea = np.count_nonzero(metvar)
    return({'Urban area': [f'{round(float(urbvararea/urbarea)*100,1)}%',round(float(urbvar.max()),2)],
            'Sydney metropolitan': [f'{round(float(metvararea/metarea)*100,1)}%',round(float(metvar.max()),2)],
            'Total domain': [f'{round(float(vararea/totarea)*100,1)}%', round(float(variable.max()),2)]})

# Get WRF indices for an ensemble
def get_indexes(caselist,start,end): #start date (sd), start hour (sh), end date (ed) and end hour (eh)
    indl = []
    for case in caselist:
        inds = match_tti(case, aest_to_utc(start[0],start[1])[1][0], aest_to_utc(start[0],start[1])[1][1])[0]
        inde = match_tti(case, aest_to_utc(end[0],end[1])[1][0], aest_to_utc(end[0],end[1])[1][1])[0]
        indl.append([inds,inde])
    return indl

# Define the urban outline in the gridded dataset
def urb_outline():
    grland = wrf.getvar(data_24_gr, 'LU_INDEX', timeidx=0, method='cat')
    urb = np.where(grland <= 49, 0, grland) #set everything that isn't urban to zero
    uniurb = np.where(urb >= 49, 1, urb) #make urban area uniform
    return uniurb

### Setting up ensembles

In [9]:
en1_17 = [data_17_crop, data_17_noeuro, data_17_nat, data_17_gr, data_17_grurb, data_17_def, data_17_defurb]
en2_17 = [data_17_crop_2, data_17_noeuro_2, data_17_nat_2, data_17_gr_2, data_17_grurb_2, data_17_def_2, data_17_defurb_2]
en3_17 = [data_17_crop_3, data_17_noeuro_3, data_17_nat_3, data_17_gr_3, data_17_grurb_3, data_17_def_3, data_17_defurb_3]
en4_17 = [data_17_crop_4, data_17_noeuro_4, data_17_nat_4, data_17_gr_4, data_17_grurb_4, data_17_def_4, data_17_defurb_4]
en5_17 = [data_17_crop_5, data_17_noeuro_5, data_17_nat_5, data_17_gr_5, data_17_grurb_5, data_17_def_5, data_17_defurb_5]

en1_18 = [data_18_crop, data_18_noeuro, data_18_nat, data_18_gr, data_18_grurb, data_18_def, data_18_defurb]
en2_18 = [data_18_crop_2, data_18_noeuro_2, data_18_nat_2, data_18_gr_2, data_18_grurb_2, data_18_def_2, data_18_defurb_2]
en3_18 = [data_18_crop_3, data_18_noeuro_3, data_18_nat_3, data_18_gr_3, data_18_grurb_3, data_18_def_3, data_18_defurb_3]
en4_18 = [data_18_crop_4, data_18_noeuro_4, data_18_nat_4, data_18_gr_4, data_18_grurb_4, data_18_def_4, data_18_defurb_4]
en5_18 = [data_18_crop_5, data_18_noeuro_5, data_18_nat_5, data_18_gr_5, data_18_grurb_5, data_18_def_5, data_18_defurb_5]

en1_20 = [data_20_crop, data_20_noeuro, data_20_nat, data_20_gr, data_20_grurb, data_20_def, data_20_defurb]
en2_20 = [data_20_crop_2, data_20_noeuro_2, data_20_nat_2, data_20_gr_2, data_20_grurb_2, data_20_def_2, data_20_defurb_2]
en3_20 = [data_20_crop_3, data_20_noeuro_3, data_20_nat_3, data_20_gr_3, data_20_grurb_3, data_20_def_3, data_20_defurb_3]
en4_20 = [data_20_crop_4, data_20_noeuro_4, data_20_nat_4, data_20_gr_4, data_20_grurb_4, data_20_def_4, data_20_defurb_4]
en5_20 = [data_20_crop_5, data_20_noeuro_5, data_20_nat_5, data_20_gr_5, data_20_grurb_5, data_20_def_5, data_20_defurb_5]

en1_24 = [data_24_crop, data_24_noeuro, data_24_nat, data_24_gr, data_24_grurb, data_24_def, data_24_defurb]
en2_24 = [data_24_crop_2, data_24_noeuro_2, data_24_nat_2, data_24_gr_2, data_24_grurb_2, data_24_def_2, data_24_defurb_2]
en3_24 = [data_24_crop_3, data_24_noeuro_3, data_24_nat_3, data_24_gr_3, data_24_grurb_3, data_24_def_3, data_24_defurb_3]
en4_24 = [data_24_crop_4, data_24_noeuro_4, data_24_nat_4, data_24_gr_4, data_24_grurb_4, data_24_def_4, data_24_defurb_4]
en5_24 = [data_24_crop_5, data_24_noeuro_5, data_24_nat_5, data_24_gr_5, data_24_grurb_5, data_24_def_5, data_24_defurb_5]

ensemble_dict = { 2017: [en1_17, en2_17, en3_17, en4_17, en5_17],
2018: [en1_18, en2_18, en3_18, en4_18, en5_18],
2020: [en1_20, en2_20, en3_20, en4_20, en5_20],
2024: [en1_24, en2_24, en3_24, en4_24, en5_24]
}

In [15]:
# Select ensemble member
en = en1_24

# Set storm date
start_date = '2024-02-13' #local date
start_hour = '00:00:00'   #local time

end_date = '2024-02-14' #local date
end_hour = '00:00:00'   #local time

# Find indices for the storm in the WRF dataset
inds = get_indexes(en,[start_date,start_hour],[end_date,end_hour])
namelist = ['Cropland','Pre-European', 'Forest', 'Gridded No-UCM', 'Gridded BEP-BEM','Default No-UCM', 'Default BEP-BEM']

# Extract surface hail from the selected ensemble
hail = []
for caseind in range(len(inds)):
    h = wrf.getvar(en[caseind], 'HAIL_MAXK1', timeidx=ALL_TIMES, method='cat')[inds[caseind][0]:inds[caseind][1]+1,:,:].max('Time')
    hail.append(h)

# Print hail statistics
print('GRID NUM COVERAGE')
for ind in range(len(hail)):
    val =np.sum(grid_number(hail[ind])[0])
    print(f"{namelist[ind]}: {val}")
print('PERCENTAGE COVERAGE')
for ind in range(len(hail)):
    val = percentcov(hail[ind])
    print(f"{namelist[ind]}: {val}")
print('MAX HAIL SIZE')
for ind in range(len(hail)):
    urb_hail = np.where(urb_outline() == 0, 0, hail)
    print(f"{namelist[ind]}: {round(hail[ind].max(['south_north','west_east']).values*100,2)} cm")

GRID NUM COVERAGE
Cropland: 12930
Pre-European: 13239
Forest: 13379
Gridded No-UCM: 12211
Gridded BEP-BEM: 11903
Default No-UCM: 12680
Default BEP-BEM: 13517
PERCENTAGE COVERAGE
Cropland: {'Urban area': ['22.7%', 0.03], 'Sydney metropolitan': ['16.2%', 0.02], 'Total domain': ['15.9%', 0.05]}
Pre-European: {'Urban area': ['28.1%', 0.03], 'Sydney metropolitan': ['25.7%', 0.03], 'Total domain': ['16.3%', 0.05]}
Forest: {'Urban area': ['26.8%', 0.04], 'Sydney metropolitan': ['20.2%', 0.03], 'Total domain': ['16.5%', 0.05]}
Gridded No-UCM: {'Urban area': ['22.6%', 0.03], 'Sydney metropolitan': ['19.3%', 0.03], 'Total domain': ['15.0%', 0.04]}
Gridded BEP-BEM: {'Urban area': ['27.4%', 0.03], 'Sydney metropolitan': ['22.7%', 0.03], 'Total domain': ['14.7%', 0.04]}
Default No-UCM: {'Urban area': ['23.3%', 0.03], 'Sydney metropolitan': ['15.1%', 0.02], 'Total domain': ['15.6%', 0.04]}
Default BEP-BEM: {'Urban area': ['26.1%', 0.03], 'Sydney metropolitan': ['19.1%', 0.02], 'Total domain': ['16.6